# NYC 311 lab: cluster notebook

Upload the whole `nyc311_lab` folder (without `data/sample/`), open this notebook from inside it, edit `DATA` below, then run the cells in order. Each stage prints a block between `===== BEGIN ... =====` and `===== END ... =====` lines; the last cell zips `outputs/` for download.

The same pipeline as one command: `python run_all.py --data "/path/to/311_batch_*.json" --work /scratch/.../work`.

In [ ]:
%pip install -q -r requirements.txt
%load_ext autoreload
%autoreload 2

## Settings
`WORK` holds the large Parquet copy (a few GB). Put it on persistent storage and keep it between runs: later runs only convert files that are new.

In [ ]:
DATA = "/path/on/cluster/311_batch_*.json"  # EDIT: where the 2,000 JSON files live
WORK = "work"                              # EDIT if home quota is small, e.g. a scratch path
OUT = "outputs"
MEMORY_LIMIT = None                         # e.g. "48GB"; None lets DuckDB use 80% of RAM

from nyc311.config import Config
from nyc311 import bundle, discover, env, ingest

args = ["--data", DATA, "--work", WORK, "--out", OUT]
if MEMORY_LIMIT:
    args += ["--memory-limit", MEMORY_LIMIT]
cfg = Config.from_args(args)
env.run(cfg);

## Stage 1: JSON to Parquet
Scans every file (checks it parses, recovers records from cut-off files), then converts new files to Parquet. Expect a few minutes for 2,000 files.

In [ ]:
ingest.run(cfg);

## Stage 2: discovery report
Describes the data's structure before any question is answered.

In [ ]:
discover.run(cfg)

## Download
Download the zip printed below and put it in `cluster_outputs/run1/` on your laptop.

In [ ]:
bundle.zip_outputs(cfg)